# Kapitel 5 – Övningsuppgifter: Dimensionsreducering

Det här är mina svar på övningsuppgifterna till kapitel 5, baserade på boken *"Lär dig AI från grunden - Tillämpad maskininlärning med Python"* (Prgomet, Johnson, Solberg, Rundberg Streuli), kapitel 5 ("Dimensionsreducering", sid 217–232), samt övningsuppgifterna från bokens GitHub-repo.

## Fråga 1 – Vad menas med curse of dimensionality?

Ju fler variabler (features) ett dataset har, desto högre dimension sägs det ha. Själva kan vi egentligen bara visualisera data upp till tre dimensioner (se Figur 5.1 i boken, som visar 0D till 3D) – vid fyra dimensioner eller fler går det inte längre att rita upp datan på ett direkt sätt, trots att vi lever i en tredimensionell värld.

Med en given mängd data gäller generellt att regressions- och klassificeringsmodeller blir bättre i snitt om man lägger till ytterligare en variabel, men det vänder vid någon punkt. Lägger man till för många variabler börjar modellerna istället prestera sämre. Var den här vändpunkten ligger beror på situationen, men tumregeln är att ju högre dimension ett dataset har, desto mer data behövs för att bygga bra modeller – fler variabler tvingar helt enkelt modellen att lära sig mer komplexa mönster.

Det som **curse of dimensionality** egentligen handlar om är att data i höga dimensioner beter sig på sätt vi inte är vana vid från två eller tre dimensioner. Boken tar upp två exempel på detta:

1. Extrema punkter blir vanligare. Väljer man en slumpmässig punkt i en tvådimensionell enhetskvadrat (1×1) är sannolikheten mindre än 0.4 % att punkten hamnar närmare än 0.001 längdenheter från en kant. Gör man samma sak i en 100-dimensionell enhetshyperkub är sannolikheten redan över 18 %, i 1000 dimensioner över 86 % och i 10 000 dimensioner större än 99.999999 %. Ju fler dimensioner desto mer sannolikt att en given punkt är extrem på något sätt.
2. Avståndet mellan punkter ökar. Genomsnittsavståndet mellan två slumpmässiga punkter i en 2D-enhetskvadrat är ungefär 0.52 längdenheter, i 3D ungefär 0.66, och i en miljon dimensioner hela ca 408 längdenheter. En ny punkt som ska prediktera hamnar alltså i snitt längre ifrån träningsdatan ju högre dimension datasetet har, vilket gör det svårare att prediktera och ökar risken för överanpassning.

Boken visar också hur antalet möjliga punkter växer exponentiellt: en linje med 10 punkter i 1D blir en kvadrat med 100 punkter i 2D, och redan vid 80 dimensioner pratar man om 10⁸⁰ punkter – ungefär lika många som det uppskattade antalet atomer i hela det observerbara universum (som jämförelse hade MNIST-datan från Avsnitt 4.4.1 "bara" 784 variabler). I teorin går problemet att lösa genom att samla in mer data så punkterna inte ligger så glest, men i praktiken är det ofta svårt eftersom mängden data som krävs växer exponentiellt med antalet dimensioner. Det är just det här som gör att man ofta vill genomföra dimensionsreducering.

## Fråga 2 – Vad är dimensionsreducering och varför görs det?

**Dimensionsreducering** går ut på att man transformerar ett dataset till ett lägre antal dimensioner, alltså färre variabler rent praktiskt. Grejen är att de nya variablerna som skapas oftast inte går att tolka på samma sätt som de ursprungliga – information försvinner på vägen, och det är priset man betalar för att göra dimensionsreducering över huvud taget.

Det är bara de oberoende variablerna (x) som transformeras – man behöver ingen beroende variabel (y), vilket är anledningen till att dimensionsreducering räknas som icke-väglett lärande. Det gäller även när man använder t.ex. PCA som ett försteg innan man tränar en modell från väglett lärande, som logistisk regression eller ett beslutsträd (se Avsnitt 5.3.1) – själva reduceringen är ändå icke-väglett lärande, oavsett vad man gör med datan efteråt.

Enligt Avsnitt 5.2.1 i boken ("Effekter av dimensionsreducering") brukar man göra det här av en eller flera av följande anledningar:

- Man vill minska tiden det tar att träna en modell, till exempel om en modell måste tränas om varje natt och bli klar inom en viss tid.
- Ibland förbättras faktiskt en modells prediktionsförmåga – även om det oftast blir tvärtom eftersom information går förlorad, kan dimensionsreducering ibland ta bort onödigt brus så att modellen inte överanpassar lika mycket.
- Man vill kunna visualisera datan. Genom att reducera ett högdimensionellt dataset till två eller tre dimensioner går det att rita upp (se Figur 5.2 och Figur 5.3 i boken), även om det då är en transformerad version av datan och tolkningen av bilden kan vara knepig.

Om en dimensionsreducering gör att modellen presterar sämre måste man väga tid mot prediktionsförmåga. Boken tar ett exempel med en klassificeringsmodell som tar 15 timmar att träna och når 70 % accuracy, jämfört med 8 timmars träning och 65 % accuracy efter dimensionsreducering. Om det är värt det eller inte får avgöras från fall till fall.

## Fråga 3 – Förklara översiktligt hur PCA fungerar (Figur 5.4, sid 224)

**PCA**, principal component analysis eller principalkomponentanalys, är den vanligaste metoden för dimensionsreducering.

Grundtanken är att man projicerar punkterna i datan ned på ett lägre dimensionellt hyperplan (en generalisering av ett plan – i 2D är det en linje, i 3D ett vanligt plan, och i ett n-dimensionellt rum har hyperplanet n-1 dimensioner). Frågan blir förstås hur man väljer det här planet. PCA väljer det hyperplan som behåller högst andel varians från den ursprungliga datan, det vill säga det plan som bevarar så mycket information som möjligt.

När jag tittar på figur 5.4 (sid 224) ser jag ett dataset i 2D med variablerna x₁ och x₂ i vänsterbilden, och tre olika linjer inritade – en heldragen, en punktmarkerad och en sträckad. Var och en av dem är en tänkbar 1-dimensionell projektionsyta (motsvarigheten till ett hyperplan i 2D). I högerbilden ser man hur punkterna hamnar efter att de projicerats ned på var och en av de tre linjerna (de tre panelerna uppifrån och ned hör ihop med heldragen, sträckad respektive punktmarkerad linje). Det syns ganska tydligt att projektionen på den heldragna linjen sprider ut punkterna mest och alltså behåller mest varians, den sträckade linjen behåller lite mindre, och den punktmarkerade linjen klumpar ihop punkterna mest och behåller minst varians.

Eftersom PCA vill behålla så mycket varians som möjligt är det den heldragna linjen som datan skulle projiceras ned på om man reducerade till 1 dimension. I figuren kallas den C₁, den första principalkomponenten – alltså den axel som fångar störst andel av variansen i den ursprungliga datan. Den behöver inte alls ligga i linje med x₁ eller x₂, utan pekar i den riktning där punkterna sprider ut sig som mest.

Nästa steg är att hitta C₂, som ska fånga näst störst andel av variansen, med kravet att den måste vara ortogonal (vinkelrät) mot C₁. Det som är lite klurigt här är att det i Figur 5.4 faktiskt är den punktmarkerade linjen som blir C₂, trots att den sträckade linjen egentligen behåller mer varians – men den sträckade linjen är inte ortogonal mot C₁ och kan därför inte väljas som andra principalkomponent.

Man fortsätter så här tills man har lika många nya axlar som det fanns dimensioner i den ursprungliga datan, där varje ny axel måste vara ortogonal mot alla tidigare axlar. Varje sådan axel kallas en principalkomponent (PK). Till sist väljer man hur många av dem man faktiskt vill behålla vid projektionen, till exempel genom att bestämma hur stor andel av variansen man vill spara (ofta 95 %) – kanske räcker de fyra första av tio principalkomponenter för att fånga 95 % av variansen, och då reducerar man datan från tio till fyra dimensioner.

Något som är värt att komma ihåg (se även Fråga 6) är att de nya variablerna PCA skapar, som z₁ i Figur 5.4, i regel saknar en direkt tolkning kopplad till de ursprungliga variablerna. Om x₁ och x₂ i exemplet hade betecknat till exempel vikt och längd, har den projicerade variabeln z₁ ingen sådan tydlig innebörd längre – den tolkningen försvinner på vägen.

## Fråga 5 – Resonemangsfråga: Stina och Kalle om prediktioner och tid

Jag håller med Kalle. Målet med en modell är förstås ofta att prediktera så bra som möjligt, men det stämmer inte att det är det enda man bryr sig om inom maskininlärning – tid är minst lika viktigt, precis som boken lyfter fram i diskussionen om dimensionsreducering (Avsnitt 5.2.1, "Effekter av dimensionsreducering").

Ta träningstiden som exempel: om en modell behöver tränas om varje natt, vilket boken faktiskt nämner som ett skäl till att göra dimensionsreducering, måste den bli klar inom utsatt tid oavsett hur bra en långsammare och mer komplex modell hade kunnat prestera. Samma sak gäller prediktionstid – vissa tillämpningar kräver svar i (nära) realtid, som självkörande fordon eller bedrägeriupptäckt vid kortköp, och då kan en modell som är lite sämre men mycket snabbare vara att föredra framför den som ger bäst möjliga prediktioner.

Boken tar upp precis den här avvägningen: en klassificeringsmodell som tar 15 timmar att träna och når 70 % accuracy kan efter dimensionsreducering ta 8 timmar men bara nå 65 % accuracy. Om det är värt det beror helt på situationen, så det finns inget universellt svar – man måste alltid väga prediktionsförmåga mot tid utifrån sammanhanget. Stinas påstående stämmer egentligen bara om man bortser från sånt som beräkningsresurser och tidskrav, och det är sällan rimligt i verkliga tillämpningar.

## Fråga 6 – Resonemangsfråga: Tolkning av variabler efter PCA

Efter en PCA försvinner i regel den direkta tolkningen av variablerna. De nya variablerna, principalkomponenterna (t.ex. z₁ och z₂), är kombinationer av de ursprungliga variablerna och är valda för att maximera behållen varians, men de motsvarar inte längre någon konkret, namngiven egenskap i datan på samma sätt som innan.

Boken visar det här med Figur 5.4 (se Fråga 3): om x₁ och x₂ ursprungligen betecknade en persons vikt och längd, har den nya variabeln z₁ som man får efter projektionen på C₁ ingen direkt koppling till vare sig vikt eller längd. Det är bara en abstrakt axel som råkar fånga mest varians i datan, inte en mätbar storhet med egen mening.

Det här är egentligen priset man betalar för dimensionsreducering (se Fråga 2): man vinner enklare och snabbare modeller och möjligheten att visualisera datan, men förlorar chansen att direkt förklara varför en modell predikterar som den gör i termer av de ursprungliga variablerna. Det gör PCA mindre lämpligt när **förklarbarhet** (interpretability) är viktigt, till exempel om man behöver kunna motivera för en beslutsfattare eller kund exakt vilka faktorer som ligger bakom en viss prediktion.

## Fråga 8 – Koduppgift: Förklara vad koden gör

Koden nedan motsvarar i princip bokens eget exempel i Avsnitt 5.3 (sid 227–228), där `.inverse_transform()` demonstreras med ett slumpmässigt dataset med 3 kolumner som reduceras till 2 dimensioner.

Går man igenom raderna en och en:

- `X = np.random.rand(1000, 3)` skapar ett slumpmässigt dataset med 1000 rader och 3 kolumner (värden mellan 0 och 1) – det här är den ursprungliga, tredimensionella datan.
- `pca = PCA(n_components=2)` skapar en PCA-modell med hyperparametern `n_components=2`, alltså väljer man manuellt att behålla 2 principalkomponenter istället för att ange en andel varians att spara (jämför med `PCA(n_components=0.7)` i bokens tidigare exempel, sid 225).
- `X2D = pca.fit_transform(X)` tränar PCA-modellen på `X` (hittar de axlar som behåller mest varians, se Fråga 3) och transformerar samtidigt datan till den nya, tvådimensionella rymden. `X2D` får därmed formen (1000, 2).
- `X3D_inv = pca.inverse_transform(X2D)` gör motsatsen – tar den dimensionsreducerade datan och försöker återskapa den ursprungliga, tredimensionella datan.
- `np.allclose(X3D_inv, X)` jämför om `X3D_inv` och den ursprungliga `X` är nästan identiska, elementvis, och skriver ut ett booleskt värde.

Varför blir resultatet `False`? Precis som boken påpekar (sid 227–228) går information oåterkalleligt förlorad när man går från 3 till 2 dimensioner med PCA – den tredje principalkomponenten, som stod för en mindre andel av variansen, kastas ju bort. `.inverse_transform()` kan därför bara approximera den ursprungliga datan utifrån de 2 dimensioner som finns kvar, inte återskapa exakta värden, eftersom informationen i den bortkastade tredje komponenten är borta för gott. Det är samma "pris" för dimensionsreducering som beskrivs i Avsnitt 5.2: de nya variablerna kan varken tolkas eller återskapas exakt (se även Fråga 2 och Fråga 6).

Värt att notera är att eftersom `X` genereras slumpmässigt (`np.random.rand`, utan fröfixering) kommer de exakta talen i utskriften nedan att skilja sig åt varje gång koden körs – men `np.allclose` blir så gott som alltid `False`, av samma anledning som ovan.

In [1]:
import numpy as np
from sklearn.decomposition import PCA

# Creating a dataset with 3 features/columns
X = np.random.rand(1000, 3)
print(X[0:5])
# Reducing the data to 2 dimensions
pca = PCA(n_components=2)
X2D = pca.fit_transform(X)
print(X2D[0:5])

# "Recreating" the data to 3 dimensions
X3D_inv = pca.inverse_transform(X2D)
# Not exactly equal since some information was lost in the transformation
print(np.allclose(X3D_inv, X))

[[0.31771582 0.29252372 0.53439967]
 [0.78364128 0.37358156 0.13731325]
 [0.52005972 0.0151896  0.54987077]
 [0.3858919  0.98595657 0.72490445]
 [0.67183871 0.48040915 0.20923889]]
[[ 0.00285663 -0.1865712 ]
 [ 0.11830597 -0.22887672]
 [ 0.320474   -0.31180602]
 [-0.31429094  0.45020528]
 [ 0.00243202 -0.14281934]]
False


## Fråga 9 – Koduppgift: PCA på car_price_dataset innan modellering

*Notera: `car_price_dataset.csv` (från kapitel 3) finns inte tillgänglig lokalt i den här miljön och behöver hämtas från bokens GitHub-repo (https://github.com/AntonioPrgomet/ai_tillaempad_ml) eller motsvarande kapitel 3-material innan koden nedan kan köras. Kodcellen nedan är därför **oexekverad** – den visar upplägget för hur en jämförelse kan göras, snarare än ett faktiskt körresultat.*

Nedan skissas hur man kan genomföra en PCA som förbehandlingssteg innan en regressionsmodell tränas på `car_price_dataset.csv` (samma dataset som i kapitel 3), samt jämföra prestandan mot en baseline-modell utan PCA.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# 1. Läs in datan (från kapitel 3, hämtas från bokens GitHub-repo)
df = pd.read_csv("car_price_dataset.csv")

# Antag att "Price" är den beroende variabeln (y) och resten är features (X)
y = df["Price"]
X = df.drop(columns=["Price"])

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Förbehandling: standardisera numeriska variabler, one-hot-encoda kategoriska
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
])

# --- Baseline: utan PCA ---
baseline_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LinearRegression()),
])
baseline_pipeline.fit(X_train, y_train)
baseline_pred = baseline_pipeline.predict(X_test)
baseline_rmse = mean_squared_error(y_test, baseline_pred, squared=False)
print(f"RMSE utan PCA: {baseline_rmse:.2f}")

# --- Med PCA (behåller 95% av variansen) som förbehandlingssteg ---
pca_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("pca", PCA(n_components=0.95)),
    ("model", LinearRegression()),
])
pca_pipeline.fit(X_train, y_train)
pca_pred = pca_pipeline.predict(X_test)
pca_rmse = mean_squared_error(y_test, pca_pred, squared=False)
print(f"RMSE med PCA: {pca_rmse:.2f}")

n_components = pca_pipeline.named_steps["pca"].n_components_
n_features_transformed = pca_pipeline.named_steps["preprocess"].transform(X_train[:1]).shape[1]
print(f"Antal dimensioner efter förbehandling: {n_features_transformed} -> {n_components} efter PCA")

**Hur påverkas resultatet i praktiken?**

Utifrån vad boken beskriver i Avsnitt 5.2.1 ("Effekter av dimensionsreducering") kan man vänta sig ungefär följande när PCA görs innan modellering på `car_price_dataset.csv`:

- Snabbare modellträning. Eftersom antalet dimensioner minskar – särskilt om datan innehåller flera one-hot-encodade kategoriska variabler (t.ex. bilmärke/modell) som tillsammans ger ganska hög dimensionalitet – blir träningen generellt snabbare efter PCA.
- Ofta lite sämre prediktionsförmåga. Dimensionsreducering har ett pris: information går förlorad när datan projiceras ned till färre dimensioner (se Fråga 2, 6 och 8), vilket oftast, men inte alltid, ger ett något högre RMSE jämfört med baseline-modellen utan PCA.
- Möjligen mindre överanpassning. Om några av de ursprungliga variablerna – brus, eller starkt korrelerade one-hot-kolumner – bidrar lite till den faktiska prediktionsförmågan, kan PCA ibland faktiskt ge en modell som presterar bättre på testdata, eftersom onödigt brus filtreras bort. Det är samma resonemang som ligger bakom curse of dimensionality (fler variabler kräver mer data för att undvika överanpassning, se Fråga 1).

Som jag ser det handlar allt det här om samma avvägning mellan tid och prediktionsförmåga som i Fråga 2 och Fråga 5. `car_price_dataset` har ett relativt litet antal ursprungliga variabler jämfört med till exempel MNIST:s 784 dimensioner, så vinsten i träningstid är nog liten, medan risken att tappa användbar information (och därmed få sämre prediktioner) är ganska påtaglig. Man får helt enkelt testa empiriskt – jämföra RMSE med och utan PCA, som i kodskissen ovan – istället för att bara anta att PCA alltid är bra att ha med.